# 00 Sort Inbox Data

`data/candidates/00_inbox/` に入れたXMLを、分析用フォルダへ自動整理するNotebook。

入れるファイル:

```text
{match_id}_tracker_box_data.xml
{match_id}_tracker_box_metadata.xml
```

このNotebookを実行すると、メタデータXMLから試合日・試合ID・対戦名を読み取り、以下へコピーする。

```text
data/candidates/01_sorted/{dataset_name}/01_tracking_core/{match_id}_tracker_box_data.xml
data/candidates/01_sorted/{dataset_name}/03_player_master/{match_id}_tracker_box_metadata.xml
```

コピー後に表示される `MATCH_DATE`、`MATCH_ID`、`MATCH_LABEL` を `00_data_check.ipynb` に入れる。


In [ ]:
import re
import shutil
import unicodedata
import xml.etree.ElementTree as ET
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
INBOX_DIR = PROJECT_ROOT / 'data' / 'candidates' / '00_inbox'
SORTED_DIR = PROJECT_ROOT / 'data' / 'candidates' / '01_sorted'

INBOX_DIR.mkdir(parents=True, exist_ok=True)
SORTED_DIR.mkdir(parents=True, exist_ok=True)

INBOX_DIR, SORTED_DIR


In [ ]:
def slugify(value):
    value = unicodedata.normalize('NFKC', value).lower()
    replacements = {
        '筑波大学': 'tsukuba',
        ' vs ': '_vs_',
        ' - ': '-',
        ' ': '_',
    }
    for old, new in replacements.items():
        value = value.replace(old, new)
    value = re.sub(r'[^a-z0-9_-]+', '', value)
    value = re.sub(r'_+', '_', value).strip('_')
    return value or 'match'


def find_inbox_xmls(inbox_dir):
    xmls = sorted(inbox_dir.glob('*.xml'))
    metadata_files = [p for p in xmls if 'metadata' in p.name]
    data_files = [p for p in xmls if 'data' in p.name and 'metadata' not in p.name]
    if not metadata_files:
        raise FileNotFoundError(f'metadata XMLが見つかりません: {inbox_dir}')
    if not data_files:
        raise FileNotFoundError(f'tracker box data XMLが見つかりません: {inbox_dir}')
    if len(metadata_files) > 1 or len(data_files) > 1:
        raise ValueError('00_inbox には1試合分の data XML と metadata XML だけを入れてください。')
    return data_files[0], metadata_files[0]


def read_match_info(metadata_xml_path):
    root = ET.parse(metadata_xml_path).getroot()
    match = root.find('.//match')
    if match is None:
        raise ValueError(f'match情報が見つかりません: {metadata_xml_path}')
    match_id = str(match.attrib['matchId'])
    match_date = match.attrib.get('matchDatetimeLocal', match.attrib.get('matchDatetime', ''))[:10]
    match_label = slugify(match.attrib.get('matchTitle', 'match'))
    dataset_name = f'{match_date}_{match_id}_{match_label}'
    return {
        'match_id': match_id,
        'match_date': match_date,
        'match_label': match_label,
        'dataset_name': dataset_name,
    }


def copy_match_files(data_xml_path, metadata_xml_path, match_info):
    dataset_dir = SORTED_DIR / match_info['dataset_name']
    tracking_dir = dataset_dir / '01_tracking_core'
    metadata_dir = dataset_dir / '03_player_master'
    tracking_dir.mkdir(parents=True, exist_ok=True)
    metadata_dir.mkdir(parents=True, exist_ok=True)

    match_id = match_info['match_id']
    tracking_output = tracking_dir / f'{match_id}_tracker_box_data.xml'
    metadata_output = metadata_dir / f'{match_id}_tracker_box_metadata.xml'

    shutil.copy2(data_xml_path, tracking_output)
    shutil.copy2(metadata_xml_path, metadata_output)

    return dataset_dir, tracking_output, metadata_output


In [ ]:
data_xml, metadata_xml = find_inbox_xmls(INBOX_DIR)
match_info = read_match_info(metadata_xml)
dataset_dir, tracking_output, metadata_output = copy_match_files(data_xml, metadata_xml, match_info)

print('整理完了')
print(f"DATASET_NAME = {match_info['dataset_name']}")
print(f"MATCH_DATE = '{match_info['match_date']}'")
print(f"MATCH_ID = '{match_info['match_id']}'")
print(f"MATCH_LABEL = '{match_info['match_label']}'")
print()
print(f'保存先: {dataset_dir}')
print(f'- {tracking_output}')
print(f'- {metadata_output}')
